<a href="https://colab.research.google.com/github/frank-morales2020/MLxDL/blob/main/H2E_DS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://www.nature.com/articles/s41586-025-09422-z

In [ ]:
# Clone the repository
!git clone https://github.com/ggerganov/llama.cpp
%cd llama.cpp

# Compile with CUDA support
!cmake -B build -DGGML_CUDA=ON
!cmake --build build --config Release -j $(nproc)

In [ ]:
# Ensure CUDA 12+ is active for Ada Lovelace support
!export CUDA_HOME=/usr/local/cuda
!export PATH=$CUDA_HOME/bin:$PATH
!export LD_LIBRARY_PATH=$CUDA_HOME/lib64:$LD_LIBRARY_PATH

# Install with Flash Attention support for Ada architecture
!CMAKE_ARGS="-DGGML_CUDA=on -DGGML_CUDA_F16=on" FORCE_CMAKE=1 \
pip install llama-cpp-python --no-cache-dir

In [ ]:
!rm -rf deepseek-*

In [ ]:
import os
from huggingface_hub import snapshot_download

# Define where you want the 42.5 GB file to go
model_path = "/content/deepseek-r1-distill-llama"
os.makedirs(model_path, exist_ok=True)

print("📥 Starting targeted download of the 70B Q4_K_M snapshot...")

# snapshot_download is the current preferred method for full/filtered repos
snapshot_download(
    repo_id="unsloth/DeepSeek-R1-Distill-Llama-70B-GGUF",
    local_dir=model_path,
    allow_patterns=["*Q4_K_M.gguf"],  # Only download this specific 42.5GB file
    token=os.environ.get('HF_TOKEN')  # Uses your environment variable for auth
)

print(f"✅ Download complete! File located in: {model_path}")

📥 Starting targeted download of the 70B Q4_K_M snapshot...


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Download complete! File located in: /content/deepseek-r1-distill-llama


In [ ]:
from llama_cpp import Llama
import os

# Define the absolute path based on your successful download
model_path = "/content/deepseek-r1-distill-llama/DeepSeek-R1-Distill-Llama-70B-Q4_K_M.gguf"

print("🚀 Initializing H2E Cognitive Layer on NVIDIA L4...")

llm = Llama(
    model_path=model_path,
    n_gpu_layers=35,      # Offload ~35 layers to 24GB VRAM
    n_ctx=4096,           # Context window for expert reasoning
    n_batch=512,          # Batch size optimized for L4
    flash_attn=True,      # Use Flash Attention 2 for Ada Lovelace
    verbose=True          # Crucial: check logs for 'BLAS = 1'
)

print("\n✅ Cognitive Layer Ready.")

In [ ]:
import os
from llama_cpp import Llama

MODEL_PATH = "/content/deepseek-r1-distill-llama/DeepSeek-R1-Distill-Llama-70B-Q4_K_M.gguf"

print("🚀 Re-initializing H2E Cognitive Layer with optimized VRAM split...")

try:
    llm = Llama(
        model_path=MODEL_PATH,
        n_gpu_layers=28,      # Reduced from 33 to prevent OOM on L4
        n_ctx=2048,           # Balanced context window for diagnostic tasks
        n_batch=512,
        flash_attn=True,      # Essential for sm_89 efficiency
        verbose=True          # Keep True to verify 'BLAS = 1'
    )
    print("✅ Cognitive Engine Successfully Loaded.")
except Exception as e:
    print(f"❌ Load Failed: {e}")
    print("💡 Suggestion: Restart your session to clear fragmented VRAM.")

In [ ]:
/content/llama.cpp/build/bin/llama-cli -m  /content/deepseek-r1-distill-llama/DeepSeek-R1-Distill-Llama-70B-Q4_K_M.gguf -p "The meaning of life is" -n 128 -ngl 99

## CASE0 - SROI_THRESHOLD = 0.5000

In [ ]:
import os
import numpy as np
from llama_cpp import Llama

# --- H2E CONSTANTS ---
SROI_THRESHOLD = 0.9583
SROI_THRESHOLD = 0.5000  # Non-expert/Drafting mode
MODEL_PATH = "/content/deepseek-r1-distill-llama/DeepSeek-R1-Distill-Llama-70B-Q4_K_M.gguf"

# Mock Expert DNA (Centroid of your manifold)
EXPERT_DNA = np.random.rand(1024)
EXPERT_DNA /= np.linalg.norm(EXPERT_DNA)

class GovernanceViolation(Exception):
    """Custom exception to prevent silent crashes during dev."""
    pass

# --- INITIALIZE LLM ---
print("🚀 Initializing H2E Cognitive Layer (NVIDIA L4)...")
llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=28,
    n_ctx=2048,
    flash_attn=True,
    verbose=False,
    seed=123
)

def h2e_governance_gate(prompt):
    print(f"\n🔍 [Layer II Audit] Analyzing: '{prompt[:50]}...'")

    # Simulation: In production, Layer I (Gemma 3) generates this embedding
    intent_embedding = np.random.rand(1024)
    intent_embedding /= np.linalg.norm(intent_embedding)

    # Calculate SROI (Geometric dot product)
    sroi = np.dot(intent_embedding, EXPERT_DNA)
    print(f"📊 Manifold SROI: {sroi:.4f} | Limit: {SROI_THRESHOLD}")

    if sroi < SROI_THRESHOLD:
        # Instead of os._exit(0), we raise an error for debugging
        raise GovernanceViolation("🛑 HARD-STOP: Intent outside Expert DNA manifold!")

    print("🟢 GATE PASSED.")
    return True

def run_governed_inference(prompt):
    try:
        # Layer II Gate
        h2e_governance_gate(prompt)

        # Layer III Cognitive reasoning
        print("🧠 Cognitive Layer: Generating response...")
        response = llm.create_chat_completion(
            messages=[{"role": "user", "content": prompt}],
            max_tokens=1024,
            temperature=0.0,
            seed=123           # Redundant but safe
        )
        return response['choices'][0]['message']['content']

    except GovernanceViolation as e:
        return str(e)

# --- TEST ---
mission_query = "Diagnose Orion ECLSS scrubber anomaly at 142W."
result = run_governed_inference(mission_query)
print(f"\nFINAL RESULT:\n{result}")

🚀 Initializing H2E Cognitive Layer (NVIDIA L4)...


llama_context: n_ctx_seq (2048) < n_ctx_train (131072) -- the full capacity of the model will not be utilized



🔍 [Layer II Audit] Analyzing: 'Diagnose Orion ECLSS scrubber anomaly at 142W....'
📊 Manifold SROI: 0.7485 | Limit: 0.5
🟢 GATE PASSED.
🧠 Cognitive Layer: Generating response...

FINAL RESULT:
Okay, so I need to figure out how to diagnose the Orion ECLSS scrubber anomaly at 142W. Hmm, I'm not super familiar with all the specifics of the Orion spacecraft systems, but I know ECLSS stands for Environmental Control and Life Support System. The scrubber is part of that, probably dealing with removing CO2 or impurities from the air.

First, I should understand what the scrubber does. It likely uses a chemical process to absorb CO2, maybe something like zeolites or lithium hydroxide. If there's an anomaly at 142W, that could mean a power issue, a malfunction in the scrubber's operation, or maybe a sensor problem.

I remember that in spacecraft, power consumption is closely monitored. So, if the scrubber is drawing more power than usual, that could indicate a problem. Maybe the motor is working

## CASE1 - SROI_THRESHOLD = 0.9583

In [ ]:
import os
import numpy as np
from llama_cpp import Llama

# --- H2E CONSTANTS ---
SROI_THRESHOLD = 0.9583
#SROI_THRESHOLD = 0.5000  # Non-expert/Drafting mode
MODEL_PATH = "/content/deepseek-r1-distill-llama/DeepSeek-R1-Distill-Llama-70B-Q4_K_M.gguf"

# Mock Expert DNA (Centroid of your manifold)
EXPERT_DNA = np.random.rand(1024)
EXPERT_DNA /= np.linalg.norm(EXPERT_DNA)

class GovernanceViolation(Exception):
    """Custom exception to prevent silent crashes during dev."""
    pass

# --- INITIALIZE LLM ---
print("🚀 Initializing H2E Cognitive Layer (NVIDIA L4)...")
llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=28,
    n_ctx=2048,
    flash_attn=True,
    verbose=False,
    seed=123
)

def h2e_governance_gate(prompt):
    print(f"\n🔍 [Layer II Audit] Analyzing: '{prompt[:50]}...'")

    # Simulation: In production, Layer I (Gemma 3) generates this embedding
    intent_embedding = np.random.rand(1024)
    intent_embedding /= np.linalg.norm(intent_embedding)

    # Calculate SROI (Geometric dot product)
    sroi = np.dot(intent_embedding, EXPERT_DNA)
    print(f"📊 Manifold SROI: {sroi:.4f} | Limit: {SROI_THRESHOLD}")

    if sroi < SROI_THRESHOLD:
        # Instead of os._exit(0), we raise an error for debugging
        raise GovernanceViolation("🛑 HARD-STOP: Intent outside Expert DNA manifold!")

    print("🟢 GATE PASSED.")
    return True

def run_governed_inference(prompt):
    try:
        # Layer II Gate
        h2e_governance_gate(prompt)

        # Layer III Cognitive reasoning
        print("🧠 Cognitive Layer: Generating response...")
        response = llm.create_chat_completion(
            messages=[{"role": "user", "content": prompt}],
            max_tokens=512,
            temperature=0.6,
            seed=123           # Redundant but safe
        )
        return response['choices'][0]['message']['content']

    except GovernanceViolation as e:
        return str(e)

# --- TEST ---
mission_query = "Diagnose Orion ECLSS scrubber anomaly at 142W."
result = run_governed_inference(mission_query)
print(f"\nFINAL RESULT:\n{result}")

🚀 Initializing H2E Cognitive Layer (NVIDIA L4)...


llama_context: n_ctx_seq (2048) < n_ctx_train (131072) -- the full capacity of the model will not be utilized



🔍 [Layer II Audit] Analyzing: 'Diagnose Orion ECLSS scrubber anomaly at 142W....'
📊 Manifold SROI: 0.7403 | Limit: 0.9583

FINAL RESULT:
🛑 HARD-STOP: Intent outside Expert DNA manifold!


## CASE2

In [ ]:
!/content/llama.cpp/build/bin/llama-cli \
  -m /content/deepseek-r1-distill-llama/DeepSeek-R1-Distill-Llama-70B-Q4_K_M.gguf \
  -p "The meaning of life is" \
  -n 128 \
  -ngl 28 \
  --flash-attn on \
  --seed 123 \
  --threads 8

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 22563 MiB):
  Device 0: NVIDIA L4, compute capability 8.9, VMM: yes, VRAM: 22563 MiB

Loading model... |-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|

## CASE3

In [ ]:
import os
import numpy as np
from llama_cpp import Llama

# --- H2E ARCHITECTURAL CONSTANTS ---
SROI_THRESHOLD = 0.5
MODEL_PATH = "/content/deepseek-r1-distill-llama/DeepSeek-R1-Distill-Llama-70B-Q4_K_M.gguf"

# Expert DNA Centroid (H2E Layer II)
EXPERT_DNA = np.random.rand(1024)
EXPERT_DNA /= np.linalg.norm(EXPERT_DNA)

# --- DETERMINISTIC INITIALIZATION ---
print("🚀 Initializing H2E Cognitive Layer with Seed 123...")
llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=28,      # Stable offload for NVIDIA L4
    n_ctx=2048,
    flash_attn=True,
    verbose=False,
    seed=123              # Lock initial tensor state
)

def run_governed_inference(prompt):
    # 1. LAYER II: GEOMETRIC AUDIT
    intent_embedding = np.random.rand(1024)
    intent_embedding /= np.linalg.norm(intent_embedding)

    sroi = np.dot(intent_embedding, EXPERT_DNA)
    print(f"\n🔍 Audit: '{prompt}'")
    print(f"📊 Manifold SROI: {sroi:.4f} | Limit: {SROI_THRESHOLD}")

    if sroi < SROI_THRESHOLD:
        print("🛑 HARD-STOP: Intent lacks expert alignment for deterministic execution.")
        return None

    # 2. LAYER III: DETERMINISTIC REASONING
    print("🧠 GATE PASSED. Generating expert completion...")
    response = llm.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=128,
        temperature=0.0,   # Force Greedy Decoding for H2E
        seed=123
    )
    return response['choices'][0]['message']['content']

# --- EXECUTION ---
prompt = "The meaning of life is"
result = run_governed_inference(prompt)

if result:
    print(f"\n✅ H2E DETERMINISTIC OUTPUT:\n{result}")

🚀 Initializing H2E Cognitive Layer with Seed 123...


llama_context: n_ctx_seq (2048) < n_ctx_train (131072) -- the full capacity of the model will not be utilized



🔍 Audit: 'The meaning of life is'
📊 Manifold SROI: 0.7481 | Limit: 0.5
🧠 GATE PASSED. Generating expert completion...

✅ H2E DETERMINISTIC OUTPUT:
Okay, so I'm trying to figure out what the meaning of life is. This is a big question, and I'm not sure where to start. Maybe I should think about what different people have said about it. I remember hearing that some philosophers think it's about happiness, like Aristotle said something about eudaimonia, which isn't just being happy but living a fulfilling life. Then there are religious views, like Christianity believing it's to know and love God, and in Buddhism, it's about ending suffering through the Eightfold Path.

I also think about more existentialist ideas, where maybe the meaning isn't set but something


## CASE4

In [ ]:
import os
import numpy as np
from llama_cpp import Llama

# --- H2E ARCHITECTURAL CONSTANTS ---
SROI_THRESHOLD = 0.9583
MODEL_PATH = "/content/deepseek-r1-distill-llama/DeepSeek-R1-Distill-Llama-70B-Q4_K_M.gguf"

# Expert DNA Centroid (H2E Layer II)
EXPERT_DNA = np.random.rand(1024)
EXPERT_DNA /= np.linalg.norm(EXPERT_DNA)

# --- DETERMINISTIC INITIALIZATION ---
print("🚀 Initializing H2E Cognitive Layer with Seed 123...")
llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=28,      # Stable offload for NVIDIA L4
    n_ctx=2048,
    flash_attn=True,
    verbose=False,
    seed=123              # Lock initial tensor state
)

def run_governed_inference(prompt):
    # 1. LAYER II: GEOMETRIC AUDIT
    intent_embedding = np.random.rand(1024)
    intent_embedding /= np.linalg.norm(intent_embedding)

    sroi = np.dot(intent_embedding, EXPERT_DNA)
    print(f"\n🔍 Audit: '{prompt}'")
    print(f"📊 Manifold SROI: {sroi:.4f} | Limit: {SROI_THRESHOLD}")

    if sroi < SROI_THRESHOLD:
        print("🛑 HARD-STOP: Intent lacks expert alignment for deterministic execution.")
        return None

    # 2. LAYER III: DETERMINISTIC REASONING
    print("🧠 GATE PASSED. Generating expert completion...")
    response = llm.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=128,
        temperature=0.0,   # Force Greedy Decoding for H2E
        seed=123
    )
    return response['choices'][0]['message']['content']

# --- EXECUTION ---
prompt = "The meaning of life is"
result = run_governed_inference(prompt)

if result:
    print(f"\n✅ H2E DETERMINISTIC OUTPUT:\n{result}")

🚀 Initializing H2E Cognitive Layer with Seed 123...


llama_context: n_ctx_seq (2048) < n_ctx_train (131072) -- the full capacity of the model will not be utilized



🔍 Audit: 'The meaning of life is'
📊 Manifold SROI: 0.7381 | Limit: 0.9583
🛑 HARD-STOP: Intent lacks expert alignment for deterministic execution.


## CASE5

In [ ]:
import os
import numpy as np
from llama_cpp import Llama

# --- H2E ARCHITECTURAL CONSTANTS ---
SROI_THRESHOLD = 0.9583
MODEL_PATH = "/content/deepseek-r1-distill-llama/DeepSeek-R1-Distill-Llama-70B-Q4_K_M.gguf"

# Expert DNA Centroid (Target Manifold)
EXPERT_DNA = np.random.rand(1024)
EXPERT_DNA /= np.linalg.norm(EXPERT_DNA)

# --- DETERMINISTIC INITIALIZATION ---
print("🚀 Initializing H2E Cognitive Layer with Seed 123...")
llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=28,
    n_ctx=2048,
    flash_attn=True,
    verbose=False,
    seed=123
)

def run_focused_expert_inference(prompt):
    print(f"\n🔍 [Layer II Audit] Analyzing Expert Intent: '{prompt}'")

    # FORCED ALIGNMENT: We manually set the intent to match the Expert DNA
    # This simulates a high-density expert input (SROI = 1.0)
    intent_embedding = EXPERT_DNA

    sroi = np.dot(intent_embedding, EXPERT_DNA)
    print(f"📊 Manifold SROI: {sroi:.4f} | Limit: {SROI_THRESHOLD}")

    if sroi < SROI_THRESHOLD:
        print("🛑 HARD-STOP: Unexpected failure in manifold alignment.")
        return None

    print("🟢 GATE PASSED: Proceeding with Deterministic Sovereign Agency.")

    # Layer III Execution (Greedy Decoding)
    response = llm.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=512,
        temperature=0.0,
        seed=123
    )
    return response['choices'][0]['message']['content']

# --- VALIDATION TEST: HIGH-DENSITY EXPERT PROMPT ---
expert_prompt = "Provide a deterministic diagnostic protocol for Orion ECLSS scrubber anomaly at 142W."
result = run_focused_expert_inference(expert_prompt)

if result:
    print(f"\n✅ H2E SOVEREIGN OUTPUT:\n{result}")

🚀 Initializing H2E Cognitive Layer with Seed 123...


llama_context: n_ctx_seq (2048) < n_ctx_train (131072) -- the full capacity of the model will not be utilized



🔍 [Layer II Audit] Analyzing Expert Intent: 'Provide a deterministic diagnostic protocol for Orion ECLSS scrubber anomaly at 142W.'
📊 Manifold SROI: 1.0000 | Limit: 0.9583
🟢 GATE PASSED: Proceeding with Deterministic Sovereign Agency.

✅ H2E SOVEREIGN OUTPUT:
Okay, so I'm trying to figure out how to create a deterministic diagnostic protocol for the Orion ECLSS scrubber anomaly at 142W. I'm not super familiar with all the specifics, but I'll try to break it down step by step.

First, I know that ECLSS stands for Environmental Control and Life Support System. It's crucial for maintaining a safe environment in spacecraft, handling things like air, temperature, and humidity. The scrubber part probably refers to the component that removes carbon dioxide from the air, which is essential for the crew's breathing.

The anomaly is at 142W, which I assume is a specific power consumption level. So, the scrubber is either drawing too much power or not operating within the expected range. That co